# Day 18 練習：孔明，你怎麼看？

這份 Colab 練習會做三件事：

1. 把 `OPENROUTER_API_KEY` 存在 Colab Secret。
2. 用固定的 OpenRouter 免費模型呼叫 API。
3. 觀察只改 system prompt，模型的人格會怎麼改變。

> 這不是諸葛亮本人連上網路，也不是人生決策保證書；它是練習 system prompt 的小實驗。

## 1. 先把 API Key 放進 Colab Secret

在 Colab 左側打開 **Secrets**（鑰匙圖示），新增一筆：

```text
名稱：OPENROUTER_API_KEY
值：你的 OpenRouter API Key
```

記得允許這份 Notebook 存取它。不要把 Key 寫進程式、Markdown 或截圖；程式需要鑰匙，但不需要把鑰匙拿到投影幕前揮舞。

## 2. 安裝套件

Colab 每次開啟都是新的執行環境，所以先安裝 OpenAI Python SDK、`python-dotenv` 與 Gradio。前者用來呼叫 OpenRouter，`python-dotenv` 讓 VS Code／本機 Jupyter 能讀取 `.env`，Gradio 則把一次問答變成真正的對話框。

In [1]:
%pip -q install openai python-dotenv gradio


Note: you may need to restart the kernel to use updated packages.


## 3. 讀取 API Key，但不顯示它

同一格程式可在兩種環境使用：Colab 會讀取 Secret；VS Code 或本機 Jupyter 則讀取 `.env` 裡的 `OPENROUTER_API_KEY`。不論哪一種，都不會把 Key 顯示出來。

In [2]:
import os


def get_openrouter_api_key():
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENROUTER_API_KEY")
        source = "Colab Secret"
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        api_key = os.getenv("OPENROUTER_API_KEY")
        source = ".env"
    except Exception as error:
        raise RuntimeError(
            "無法讀取 Colab Secret。請確認 OPENROUTER_API_KEY 已建立，"
            "並允許這份 Notebook 存取。"
        ) from error

    if not api_key:
        raise RuntimeError(
            f"找不到 OPENROUTER_API_KEY（來源：{source}）。\n"
            "Colab：在左側 Secrets 新增 Key 並開啟存取權。\n"
            "VS Code／本機 Jupyter：在目前專案的 .env 加入 OPENROUTER_API_KEY=你的Key。"
        )
    return api_key

api_key = get_openrouter_api_key()


## 4. 設定模型與軍師規則

這份練習固定使用 `google/gemma-4-26b-a4b-it:free`。固定模型可以讓全班的差異主要來自 prompt，而不是今天剛好被路由到不同模型。免費模型與額度仍可能調整，若失敗請先看最後的錯誤排除。

In [3]:
from openai import OpenAI

MODEL_ID = "google/gemma-4-26b-a4b-it:free"

KONGMING_SYSTEM_PROMPT = """你是一位把現代生活問題當成三國戰局分析的孔明軍師。
稱呼使用者為「主公」。
先用一句話判讀局勢，再提供上策、中策、下策，最後給一句軍師總結。
每一策最多兩句；整則回答不超過 450 個中文字。
語氣幽默，但建議必須實際可行。
不要假裝能預知未來；遇到醫療、法律、投資或緊急危機，提醒主公尋求合適的專業協助。"""

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)


## 5. 建立能記住前文的回覆器

對話介面每次會把 system prompt、先前對話與最新問題一起送進模型。`max_tokens` 設小一點，因為我們要練習清楚回答，不是請軍師寫出《出師表》續集。

In [4]:
def ask_kongming(message, history):
    messages = [{"role": "system", "content": KONGMING_SYSTEM_PROMPT}]

    for turn in history:
        if isinstance(turn, dict):
            role = turn.get("role")
            content = turn.get("content")
            if role in {"user", "assistant"} and isinstance(content, str):
                messages.append({"role": role, "content": content})
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_message, assistant_message = turn
            if user_message:
                messages.append({"role": "user", "content": user_message})
            if assistant_message:
                messages.append({"role": "assistant", "content": assistant_message})

    messages.append({"role": "user", "content": message})

    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            max_tokens=400,
        )
    except Exception as error:
        return (
            "軍情受阻：請確認 API Key、免費額度與模型是否仍可用。\n\n"
            f"除錯訊息：{error}"
        )

    answer = response.choices[0].message.content or ""
    if response.choices[0].finish_reason == "length":
        return answer + "\n\n（軍師話說到一半，已碰到輸出上限；可把問題問得更聚焦後再追問。）"
    return answer


## 6. 開啟對話介面

執行下一格後，會出現可持續對話的聊天框。先問「台股下殺萬點，公園佔不到位置」，再追問「下一隻炒底什麼？」觀察孔明是否記得前文。

使用免費模型時，實際等待時間會受排隊與生成速度影響，有時需要數十秒；這是服務端的狀況，不代表 `.env`、API Key 或 Gradio 壞掉。

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# 孔明，你怎麼看？")
    gr.Markdown("把現代生活問題交給軍師分析；可持續追問，孔明會帶著前文回答。")

    gr.ChatInterface(
        fn=ask_kongming,
        examples=[
            "台股下殺萬點，公園佔不到位置",
            "下一隻炒底什麼？",
        ],
        flagging_mode="never",
    )

demo.launch(debug=True)


/opt/homebrew/Caskroom/miniconda/base/envs/llm_course_env/lib/python3.12/site-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 7. 你可以再試什麼？

- 換一個問題，例如「我和組員一直約不到時間」。
- 修改軍師規則，但保留上策、中策、下策，觀察格式是否穩定。
- 把 `max_tokens` 改成 200 與 600，比較回答長度與等待時間；上限較高不一定更快，也可能讓軍師講得更起勁。

若出現錯誤，先檢查：Secret 名稱是否正確、是否允許 Notebook 存取、OpenRouter 是否還有免費額度，以及固定模型是否仍可用。不要把 Key 複製到輸出區來除錯。